# Exploratory Data Analysis
## Synthetic Sleep Environment Dataset
### TECHIN 513 Final Project — Team 7

We explore the generated dataset: signal characteristics, label distributions,
feature statistics, and cross-signal relationships.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import seed_everything, ensure_dirs
from src.data_generation import generate_dataset
from src.feature_extraction import build_feature_matrix, FEATURE_NAMES

seed_everything(42)
ensure_dirs()
sessions = generate_dataset(n_sessions=200, global_seed=42)
X, y, feat_names, label_names, meta = build_feature_matrix(sessions)
df = pd.DataFrame(X, columns=feat_names)
for i, ln in enumerate(label_names):
    df[ln] = y[:, i]
for k in ('session_id', 'season', 'quality_class'):
    df[k] = [m[k] for m in meta]
print('Shape:', df.shape)
df[label_names].describe()

## Label Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), label_names):
    df[col].hist(bins=30, ax=ax, color='#9467bd', alpha=0.7, edgecolor='white')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel(col)
plt.suptitle('Sleep Quality Label Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Signal Statistics by Quality Class

In [ ]:
df.groupby('quality_class')[['temp_mean', 'light_n_events', 'noise_n_events',
                             'sleep_efficiency', 'awakenings']].agg(['mean', 'std']).round(3)

## Temperature FFT Spectrum

In [ ]:
from src.signal_processing import compute_fft_spectrum
from src.utils import SAMPLE_RATE_MIN
sess = sessions[0]
freqs, psd = compute_fft_spectrum(sess['temperature'])
plt.figure(figsize=(10, 4))
plt.semilogy(freqs[1:], psd[1:], color='#d62728')
hvac_f = 1.0/90.0
plt.axvline(hvac_f, color='grey', linestyle='--', label='HVAC (~90 min period)')
plt.xlabel('Frequency (cycles per minute)')
plt.ylabel('PSD')
plt.title('Temperature Power Spectrum')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Session Example: All Four Signals

In [ ]:
t_hours = sessions[5]['t_min'] / 60.0
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for ax, (key, label, color) in zip(axes, [
    ('temperature', 'Temperature (C)', '#d62728'),
    ('light', 'Light (lux)', '#ff7f0e'),
    ('humidity', 'Humidity (%)', '#1f77b4'),
    ('noise', 'Noise (dB)', '#2ca02c'),
]):
    ax.plot(t_hours, sessions[5][key], color=color, lw=1.2)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Hours into sleep session')
plt.suptitle('Example Session — All Signals', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()